# FMPE Normalization Benchmark — Analysis

Load pre-computed results from `results/` and visualise all figures.
No GPU or JAX required — this notebook is pure matplotlib.

**Run the experiments first:**
```bash
python main.py                                          # local, small model
python main.py --hidden 128 512 1024 512 128 --batch_size 256  # GPU / Colab
```


## 1 — Load results


In [ ]:
from utils import (
    load_results,
    plot_results,
    plot_lipschitz_curves,
    plot_nfe_vs_tolerance_from_data,
)

RESULTS_DIR = "results"
TASKS       = ["two_moons", "slcp"]

data = {}
for task in TASKS:
    metrics, lip, nfe_tol, config = load_results(RESULTS_DIR, task)
    data[task] = dict(metrics=metrics, lip=lip, nfe_tol=nfe_tol)
    print(f"{task}: budgets={config['budgets']}, seeds={config['seeds']}, hidden={config['hidden_sizes']}") 


## 2 — Benchmark results: C2ST / NFE / inference time

3 × 3 grid per task: rows = normalization, columns = C2ST / NFE / inference time.
Lines = ODE tolerances; shading = min–max over seeds.


In [ ]:
for task in TASKS:
    print(f"\n── {task} ──")
    plot_results(data[task]["metrics"], config["budgets"], task_name=task)


## 3 — Lipschitz constant over training

Empirical $\hat{L}$ of $v_\phi(t{=}0.5, \cdot, x_\text{obs})$ throughout training.
Spectral norm keeps $\hat{L}\approx 1$ by construction.


In [ ]:
for task in TASKS:
    print(f"\n── {task} ──")
    plot_lipschitz_curves(data[task]["lip"], config["budgets"], eval_freq=config["eval_freq"])


## 4 — ODE stiffness: NFE vs tolerance

Sweep from loose ($10^{-2}$) to tight ($10^{-7}$) tolerance.
A Lipschitz-bounded vector field → smoother ODE → NFE grows more slowly.


In [ ]:
for task in TASKS:
    print(f"\n── {task} ──")
    plot_nfe_vs_tolerance_from_data(data[task]["nfe_tol"], task_name=task)


## 5 — Cross-task comparison (100 k budget)

Side-by-side C2ST at the largest budget for Two Moons vs SLCP.
Tests whether normalization effects generalise beyond the simple 2-D setting.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

norm_colors = {"none": "#d62728", "layer": "#1f77b4", "spectral": "#2ca02c"}
norm_labels = {"none": "No norm", "layer": "Layer Norm", "spectral": "Spectral Norm"}
tol = 1e-7
budget = max(config["budgets"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, task in zip(axes, TASKS):
    metrics = data[task]["metrics"]
    norms   = ["none", "layer", "spectral"]
    means   = [np.mean(metrics[n][f"c2st_{tol}"][budget]) for n in norms]
    errs    = [(np.mean(metrics[n][f"c2st_{tol}"][budget]) - np.min(metrics[n][f"c2st_{tol}"][budget]),
                np.max(metrics[n][f"c2st_{tol}"][budget]) - np.mean(metrics[n][f"c2st_{tol}"][budget]))
               for n in norms]
    lo  = [e[0] for e in errs]
    hi  = [e[1] for e in errs]

    bars = ax.bar(norms, means, color=[norm_colors[n] for n in norms],
                  width=0.5, alpha=0.85)
    ax.errorbar(norms, means, yerr=[lo, hi], fmt="none",
                color="black", capsize=5, linewidth=1.5)
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1.5, label="ideal (0.5)")
    ax.set_title(f"{task}  |  budget={budget:,}  |  tol=1e-7")
    ax.set_ylabel("C2ST" if task == TASKS[0] else "")
    ax.set_ylim(0.4, 1.05)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.suptitle("C2ST at 100 k budget — Two Moons vs SLCP\nBars = mean over seeds; error = min–max", fontsize=13)
plt.tight_layout()
plt.show()


## 6 — Discussion

### Posterior quality (C2ST)

At **100 k simulations** the simulation budget dominates and all three normalizations
tend to converge to C2ST ≈ 0.5.  Differences are most pronounced at **1 k simulations**.

For **SLCP**, the C2ST gap between budgets is expected to be larger: the posterior is more
complex and the model needs more data to cover it accurately.

### Lipschitz dynamics

- **Spectral norm**: $\hat{L}$ stays near 1 throughout training — bounded by construction.
- **No norm**: $\hat{L}$ grows as the network fits the data, making the ODE progressively stiffer.
- **Layer norm**: intermediate; stabilises gradient scale but does not constrain the spectral radius.

### NFE vs tolerance

The NFE-vs-tolerance curve is the direct empirical test of the spectral-norm claim.
A Lipschitz-bounded $v_\phi$ produces a smoother ODE: the Dopri5 adaptive solver can
take larger steps without exceeding the local error threshold, so NFE grows more slowly
as the tolerance tightens.

### Practical take-away

| Normalization | Primary benefit | When to prefer |
|---------------|----------------|---------------|
| **Layer Norm** | Training stability at low budget | Simulations are expensive |
| **Spectral Norm** | Faster inference (lower NFE) | Many inference calls needed |
| **None** | Fastest training iteration | Large budget, loose tolerance |
